# Step 17 — Evaluating Speed, Accuracy, and Cost

**Optional.** Builds on [Step 16](step_16_design_patterns.ipynb)'s Pattern 7 (Evaluator-Optimizer) and feeds `REPORT.md`'s Section 5.2 (Performance). CrewAI has no built-in "evaluate my agent" command — this notebook builds a small, reusable evaluation harness out of three things: two CrewAI already tracks natively (timing, token usage), and one you build yourself (an LLM-as-judge accuracy check, reused from Step 16). Run over a small test set, the three combine into exactly the numbers Section 5.2 asks for: Goal Completion Rate, average latency, and average cost per run.

## Learning objective

By the end of this notebook, you will:

- Know where each of the three metrics actually comes from: `time.time()`, `CrewOutput.token_usage`, and a custom LLM-as-judge accuracy check
- Have read `result.token_usage`'s real fields (`total_tokens`, `prompt_tokens`, `cached_prompt_tokens`, `completion_tokens`, `successful_requests`) and converted them into an approximate dollar cost
- Have reused Step 16's evaluator-optimizer judge mechanism as a standalone batch scorer instead of an inline retry gate
- Have run a small evaluation harness over multiple test cases and produced aggregate metrics you can paste directly into `REPORT.md`'s Section 5.2
- Be able to compare your own agent's cost and latency against your Step 02 zero-shot baseline with real numbers, not impressions

## Prerequisites

- [Step 08 — Introduction to CrewAI](step_08_intro_to_crewai.ipynb) and [Step 09 — Single Agent](step_09_single_agent.ipynb) completed
- [Step 16 — Agentic Workflow Design Patterns](step_16_design_patterns.ipynb) recommended, not required — this notebook reuses Pattern 7's exact judge mechanism
- The same `.env` setup as the previous steps

## Background

None of CrewAI's built-in features tell you whether your agent is actually *good* — `tracing=True` shows you what happened, `token_usage` shows you what it cost, but neither one knows what the *right* answer was supposed to be. That's true of every agent framework, not a CrewAI gap: correctness is inherently task-specific, so an evaluation harness is something you build once per project, not something a framework can ship for you.

The three metrics below map directly onto `REPORT.md` Section 5.2's questions:

| Section 5.2 asks | This notebook measures it via |
| --- | --- |
| Goal Completion Rate | An LLM-as-judge scoring each output against a known-correct expected answer |
| Cost & latency vs. baseline | `result.token_usage` and `time.time()`, averaged across a test set |

## How this works

1. **Latency** — wrap `crew.kickoff()` in `time.time()`. `Crew(tracing=True)` (Steps 08/14–16) gives you a finer-grained breakdown per task if you need it, but a wall-clock timer is enough for an aggregate number.
2. **Cost** — `CrewOutput.token_usage` is populated automatically after every `kickoff()`, no configuration needed. Multiply `total_tokens` by your model's per-token price to get an actual dollar figure.
3. **Accuracy** — CrewAI can't check this for you. The cell below reuses [Step 16](step_16_design_patterns.ipynb) Pattern 7's `llm_judge` idea almost unchanged: instead of gating a single task's retries, it scores a batch of already-finished outputs against known-correct expected answers.
4. **The harness** — loop over a small test set, run the crew once per case, collect `(correct, elapsed, tokens)` for each, then average.

Run the cells in order — the harness at the end depends on the agent, judge function, and test set defined above it.

In [1]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

# ── The agent under test — swap this for your own team's agent when you reuse
# this harness on your own project. ───────────────────────────────────────────
qa_agent = Agent(
    role="Trivia Answerer",
    goal="Answer factual questions directly and correctly",
    backstory="You give short, confident, factual answers — no hedging, no extra commentary.",
    llm=llm,
    verbose=False,
)

# ── A small, known-correct test set — swap for your own project's test cases ─
test_cases = [
    {"question": "What is the capital of France?", "expected": "Paris"},
    {"question": "What is 12 times 7?", "expected": "84"},
    {"question": "Who wrote the play Romeo and Juliet?", "expected": "William Shakespeare"},
    {"question": "What is the chemical symbol for gold?", "expected": "Au"},
]


### The accuracy check — an LLM judge, reused from Step 16

Exact string matching would fail on harmless wording differences ("Au" vs. "the symbol Au" vs. "Au (gold)"), so — same as Step 16 Pattern 7 — a small judge `Crew` compares the agent's actual answer to the expected one and returns a plain `True`/`False`.

In [2]:
from pydantic import BaseModel

class Verdict(BaseModel):
    valid: bool
    feedback: str | None = None

def judge_answer(actual: str, expected: str) -> bool:
    judge_agent = Agent(
        role="Grader",
        goal="Judge whether an answer conveys the same fact as the expected answer",
        backstory="You ignore wording and formatting differences and focus only on factual correctness.",
        llm=llm,
    )
    judge_task = Task(
        description=(
            f"Does this answer: '{actual}' correctly convey the same fact as this "
            f"expected answer: '{expected}'? Minor wording or formatting differences "
            "are fine — only the core fact has to match."
        ),
        expected_output="A verdict on whether the answer is factually correct.",
        agent=judge_agent,
        output_pydantic=Verdict,
    )
    verdict = Crew(agents=[judge_agent], tasks=[judge_task], process=Process.sequential).kickoff().pydantic
    return verdict.valid


### The harness — run every test case, collect speed + cost + accuracy

For each test case: build a fresh `Task`, time the `kickoff()`, read `token_usage` off the result, and grade it with `judge_answer`. Nothing here is CrewAI-specific — the same three lines (`time.time()`, `result.token_usage`, a correctness check) work for any `Crew`, including your own project's.

In [3]:
# Adjust to your model's actual pricing (USD per 1M tokens) for a real cost estimate —
# this is illustrative only, not looked up from a live pricing API.
PRICE_PER_1M_TOKENS = 0.10

results = []
for case in test_cases:
    task = Task(
        description=case["question"],
        expected_output="A short, direct, factual answer.",
        agent=qa_agent,
    )
    crew = Crew(agents=[qa_agent], tasks=[task], process=Process.sequential, verbose=False)

    start = time.time()
    result = crew.kickoff()
    elapsed = time.time() - start

    correct = judge_answer(result.raw, case["expected"])

    results.append({
        "question": case["question"],
        "answer": result.raw,
        "expected": case["expected"],
        "correct": correct,
        "elapsed": elapsed,
        "tokens": result.token_usage.total_tokens,
    })

# ── Per-case detail ────────────────────────────────────────────────────────────
for r in results:
    status = "PASS" if r["correct"] else "FAIL"
    print(f"[{status}] {r['question']!r} -> {r['answer']!r} ({r['elapsed']:.2f}s, {r['tokens']} tokens)")

# ── Aggregate metrics — paste these into REPORT.md Section 5.2 ────────────────
completion_rate = sum(r["correct"] for r in results) / len(results)
avg_latency = sum(r["elapsed"] for r in results) / len(results)
avg_tokens = sum(r["tokens"] for r in results) / len(results)
avg_cost = avg_tokens / 1_000_000 * PRICE_PER_1M_TOKENS

print("\n=== Aggregate ===")
print(f"Goal Completion Rate: {completion_rate:.0%} ({sum(r['correct'] for r in results)}/{len(results)})")
print(f"Average latency:      {avg_latency:.2f}s per run")
print(f"Average tokens:       {avg_tokens:.0f} per run")
print(f"Average cost:         ${avg_cost:.5f} per run (at ${PRICE_PER_1M_TOKENS}/1M tokens)")


[PASS] 'What is the capital of France?' -> 'Paris' (0.73s, 90 tokens)
[PASS] 'What is 12 times 7?' -> '84' (0.44s, 645 tokens)
[PASS] 'Who wrote the play Romeo and Juliet?' -> 'William Shakespeare' (0.51s, 1289 tokens)
[PASS] 'What is the chemical symbol for gold?' -> 'Au' (0.51s, 2025 tokens)

=== Aggregate ===
Goal Completion Rate: 100% (4/4)
Average latency:      0.55s per run
Average tokens:       1012 per run
Average cost:         $0.00010 per run (at $0.1/1M tokens)


## Your task

1. Run the harness as-is. Does the judge ever disagree with what you'd have graded by hand? Read the flagged cases and decide whether the judge or your own instinct was right.
2. Add 2–3 test cases that you expect the agent to get *wrong* (ambiguous questions, or questions outside its likely training knowledge). Does `completion_rate` actually drop, or does the agent do better than you expected?
3. Swap `qa_agent` for one of your own team's agents, and `test_cases` for a handful of realistic inputs from your own project — this is the exact evidence `REPORT.md` Section 5.2 asks for.
4. **Baseline comparison:** rerun `test_cases` through a plain `llm.call()` (no `Agent`/`Task`/`Crew`, like [Step 02](step_02_zero_shot_prompting.ipynb)) instead of `qa_agent`, and record its latency and (if your provider exposes it) token usage. How much of the agent's extra cost/latency over the plain call is actually buying you anything, per `REPORT.md` Section 5.1's baseline-comparison question?
5. Change `PRICE_PER_1M_TOKENS` to your actual model's real published price, and multiply `avg_cost` by how many times you expect to run this in production (e.g. per day) — turns an abstract token count into a number a stakeholder would actually read in Section 5.3 (Business Value).

## Shortcomings

- The judge is itself an LLM call — it can be wrong, and it adds one extra request (and its own latency/cost) per test case that isn't part of the agent's own run. Don't count the judge's own tokens as part of the agent's cost.
- Four test cases is enough to demonstrate the harness, not enough to trust the completion rate as a real number — a credible evaluation needs a test set sized to your actual use case, ideally 15–20+ cases covering both easy and hard inputs.
- `PRICE_PER_1M_TOKENS` here is a placeholder. Real pricing varies by provider, model, and whether tokens are cached (`cached_prompt_tokens` is usually billed at a lower rate) — check your provider's actual pricing page before reporting a cost figure as fact.
- This harness runs test cases one after another. If you have many, wire it into [Step 16](step_16_design_patterns.ipynb) Pattern 5 (Parallelization) — `async_execution=True` on each case's `Task` — to speed up the evaluation run itself, though that changes what "latency per run" measures.

## Resources for further reading

- [CrewAI Crew concept docs](https://docs.crewai.com/en/concepts/crews) — `usage_metrics`/`token_usage` and other Crew-level fields
- [CrewAI Tracing docs](https://docs.crewai.com/en/observability/tracing) — the `tracing=True` dashboard, for a finer-grained latency breakdown than a single wall-clock timer
- `REPORT.md` Section 5.2 (Performance) and 5.3 (Business Value) — where these numbers actually get used

## Stretch goal

Build a second harness run using a different model (e.g. swap `MODEL` to a smaller/cheaper or larger/pricier option your provider offers) over the same `test_cases`, and compare completion rate against average cost across the two. Is the more expensive model's accuracy gain, if any, worth its cost difference for *your* specific task?

---

This notebook is optional and not part of what's graded — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for what's actually required.